# Validação da Analytics de Saúde

Este notebook consulta exclusivamente os produtos gerados em `data/analytics/`.

A validação confirma que os resultados representam a coorte de Diabetes, Hipertensão e Obesidade, respeitam o schema analítico e não publicam grupos com menos de 10 clientes.

In [6]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "data").is_dir():
    project_root = project_root.parent

analytics_path = project_root / "data" / "analytics"

target_health_conditions = {
    "Diabetes",
    "Hipertensão",
    "Obesidade",
}

minimum_group_size = 10

print(f"Diretório: {analytics_path}")
print(f"Coorte: {sorted(target_health_conditions)}")
print(f"Tamanho mínimo do grupo: {minimum_group_size}")

Diretório: C:\Users\Mileno\Downloads\Projeto LGPD\data\analytics
Coorte: ['Diabetes', 'Hipertensão', 'Obesidade']
Tamanho mínimo do grupo: 10


In [7]:
product_dimensions = {
    "customers_by_health_condition": ["condicao_saude"],
    "customers_by_state": ["condicao_saude", "estado"],
    "customers_by_age": ["condicao_saude", "faixa_etaria"],
    "customers_by_income": ["condicao_saude", "faixa_renda"],
    "customers_by_channel": ["condicao_saude", "canal_preferido"],
}

metric_columns = [
    "quantidade_clientes",
    "receita_total",
    "quantidade_compras",
    "ticket_medio",
]

products = {}
expected_columns_by_product = {}

for product_name, dimensions in product_dimensions.items():
    path = analytics_path / f"{product_name}.parquet"
    assert path.exists(), f"Produto não encontrado: {path}"

    dataframe = pd.read_parquet(path)
    expected_columns = [*dimensions, *metric_columns]

    assert list(dataframe.columns) == expected_columns, product_name
    assert not dataframe[dimensions].isna().any().any(), product_name
    assert not dataframe.duplicated(dimensions).any(), product_name
    assert (dataframe["quantidade_clientes"] >= minimum_group_size).all(), product_name
    assert (dataframe[metric_columns] >= 0).all().all(), product_name
    assert set(dataframe["condicao_saude"]) <= target_health_conditions, product_name

    products[product_name] = dataframe
    expected_columns_by_product[product_name] = expected_columns

print(f"Produtos carregados e validados: {len(products)}")
print("Dimensões compostas, grupos mínimos e métricas não negativas: OK")

Produtos carregados e validados: 5
Dimensões compostas, grupos mínimos e métricas não negativas: OK


In [8]:
health_product = products["customers_by_health_condition"]

assert set(health_product["condicao_saude"]) == target_health_conditions

print("Condições publicadas:")
display(health_product)

Condições publicadas:


,condicao_saude,quantidade_clientes,receita_total,quantidade_compras,ticket_medio
0,Diabetes,1028,25389243.68,51674,491.334979
1,Hipertensão,1064,27164546.77,53946,503.550713
2,Obesidade,1024,25494628.78,51064,499.268149


## Produtos por dimensão

As tabelas abaixo são agregadas e não contêm `customer_id`. Todas foram calculadas somente sobre a coorte de saúde configurada.

In [9]:
for product_name, dataframe in products.items():
    print(f"\n{product_name}")
    display(dataframe)


customers_by_health_condition


,condicao_saude,quantidade_clientes,receita_total,quantidade_compras,ticket_medio
0,Diabetes,1028,25389243.68,51674,491.334979
1,Hipertensão,1064,27164546.77,53946,503.550713
2,Obesidade,1024,25494628.78,51064,499.268149



customers_by_state


,condicao_saude,estado,quantidade_clientes,receita_total,quantidade_compras,ticket_medio
0,Diabetes,AC,36,891369.13,1932,461.371185
1,Diabetes,AL,40,939859.34,1982,474.197447
2,Diabetes,AM,37,924984.04,1726,535.911958
3,Diabetes,AP,40,956887.28,2024,472.770395
4,Diabetes,BA,25,563961.53,1272,443.365983
...,...,...,...,...,...,...
76,Obesidade,RS,49,1103042.91,2458,448.756269
77,Obesidade,SC,37,996328.09,1971,505.493704
78,Obesidade,SE,33,848777.72,1716,494.625711
79,Obesidade,SP,33,869330.87,1637,531.051234



customers_by_age


,condicao_saude,faixa_etaria,quantidade_clientes,receita_total,quantidade_compras,ticket_medio
0,Diabetes,18-24,104,2510634.73,5467,459.234449
1,Diabetes,25-34,149,3805885.79,7783,488.999845
2,Diabetes,35-44,145,3421087.12,7147,478.674566
3,Diabetes,45-54,162,4196410.48,8235,509.582329
4,Diabetes,55-64,151,3544204.27,7571,468.128949
5,Diabetes,65+,317,7911021.29,15471,511.345181
6,Hipertensão,18-24,110,2905987.56,5784,502.418320
7,Hipertensão,25-34,157,3906005.52,7704,507.010062
8,Hipertensão,35-44,170,4204675.38,8289,507.259667
9,Hipertensão,45-54,149,3780051.11,7451,507.321314



customers_by_income


,condicao_saude,faixa_renda,quantidade_clientes,receita_total,quantidade_compras,ticket_medio
0,Diabetes,10.001-20.000,452,11405491.19,22677,502.954147
1,Diabetes,3.001-5.000,114,2677373.23,6474,413.557805
2,Diabetes,5.001-10.000,311,7384983.88,14900,495.636502
3,Diabetes,Acima de 20.000,130,3317116.80,6480,511.900741
4,Diabetes,Até 3.000,21,604278.58,1143,528.677673
5,Hipertensão,10.001-20.000,442,11132891.71,22881,486.556169
6,Hipertensão,3.001-5.000,91,2560544.97,4615,554.830979
7,Hipertensão,5.001-10.000,343,9057597.02,17072,530.552778
8,Hipertensão,Acima de 20.000,161,3652542.15,7945,459.728402
9,Hipertensão,Até 3.000,27,760970.92,1433,531.033440



customers_by_channel


,condicao_saude,canal_preferido,quantidade_clientes,receita_total,quantidade_compras,ticket_medio
0,Diabetes,Aplicativo,261,6180286.08,12529,493.278480
1,Diabetes,Email,248,6407962.84,12814,500.075140
2,Diabetes,SMS,262,6430926.53,12825,501.436766
3,Diabetes,WhatsApp,257,6370068.23,13506,471.647285
4,Hipertensão,Aplicativo,278,7013266.79,14135,496.163197
5,Hipertensão,Email,239,6326640.12,12659,499.774083
6,Hipertensão,SMS,251,6352452.73,12513,507.668243
7,Hipertensão,WhatsApp,296,7472187.13,14639,510.430161
8,Obesidade,Aplicativo,268,6416585.81,13261,483.868925
9,Obesidade,Email,257,6236887.17,13060,477.556445


In [10]:
summary = pd.DataFrame(
    [
        {
            "produto": product_name,
            "grupos": len(dataframe),
            "clientes_publicados": int(dataframe["quantidade_clientes"].sum()),
            "receita_total": float(dataframe["receita_total"].sum()),
        }
        for product_name, dataframe in products.items()
    ]
)

display(summary)

,produto,grupos,clientes_publicados,receita_total
0,customers_by_health_condition,3,3116,78048419.23
1,customers_by_state,81,3116,78048419.23
2,customers_by_age,18,3116,78048419.23
3,customers_by_income,15,3116,78048419.23
4,customers_by_channel,12,3116,78048419.23
